# Class 3 — LP Workflow Agent: Formulate, Solve, and Write a Report

**Goal.** Build one practical LP workflow with **two tools**:

1. `solve_lp_from_expressions` — deterministic LP solve from algebraic expressions
2. `write_lp_report_markdown` — writes a `.md` report to disk

No tool comparison in this notebook. We focus on one end-to-end pattern:

- formulate LP from a word problem,
- call expression-based solver tool,
- call markdown-writer tool,
- return a final user-facing summary.

The report file must include:

- full LP formulation,
- final solution,
- business recommendation.

## 0. Setup

Same environment pattern as Part 04: `.env` at project root with `OPENAI_API_KEY`, and `cvxpy` installed in `.venv`.

In [1]:
import os
from pathlib import Path
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI
import cvxpy as cp

from agents import Agent, Runner, function_tool
from agents import ToolCallItem, ToolCallOutputItem, MessageOutputItem

PROJECT_ROOT = Path.cwd().parent
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — add it to .env at the project root"
client = OpenAI()

MODEL = "gpt-5.4-mini"
print("OK — client ready, cvxpy version:", cp.__version__)

OK — client ready, cvxpy version: 1.8.2


## 1. Problem text

We'll reuse Problem 1 so results can be sanity-checked against known values.

In [2]:
PROBLEM_1 = """
A certain plant can manufacture four products A, B, C, and D in any combination.
Each product requires time on each of three machines (in minutes per unit):

    Product   M1   M2   M3
    A         12    8    5
    B          7    9   10
    C          8    4    7
    D         10    0    3

Machine 1, 2, and 3 are available 20, 40, and 10 hours per week respectively.
Material costs are $2 each for products A and B and $1 each for C and D.
All products are competitive and any amounts made may be sold at $5, $4, $5, $5.
Variable labor costs are $4/hour for machines 1 and 2 and $3/hour for machine 3.
Find the best product mix to maximize total weekly profit.
"""

print(PROBLEM_1[:300] + "...")


A certain plant can manufacture four products A, B, C, and D in any combination.
Each product requires time on each of three machines (in minutes per unit):

    Product   M1   M2   M3
    A         12    8    5
    B          7    9   10
    C          8    4    7
    D         10    0    3

Machi...


## 2. Core solver utility (plain Python)

Write and test core logic first, then wrap it as a tool.

In [3]:
def _solve_lp_from_exprs(
    sense: Literal["maximize", "minimize"],
    variable_names: list[str],
    objective: str,
    constraints: list[str],
    constraint_names: list[str],
) -> dict:
    if len(constraints) != len(constraint_names):
        raise ValueError("constraints and constraint_names must have the same length.")

    cvx_vars = {name: cp.Variable(nonneg=True) for name in variable_names}
    safe_globals = {"__builtins__": {}}

    try:
        obj_expr = eval(objective, safe_globals, cvx_vars)
    except Exception as e:
        raise ValueError(f"could not parse objective {objective!r}: {e}") from e

    prob_obj = cp.Maximize(obj_expr) if sense == "maximize" else cp.Minimize(obj_expr)

    cvx_cons = []
    for c_str in constraints:
        try:
            cvx_cons.append(eval(c_str, safe_globals, cvx_vars))
        except Exception as e:
            raise ValueError(f"could not parse constraint {c_str!r}: {e}") from e

    problem = cp.Problem(prob_obj, cvx_cons)
    problem.solve(solver=cp.HIGHS)

    if any(cvx_vars[name].value is None for name in variable_names):
        return {
            "status": problem.status,
            "optimal_value": None,
            "variable_values": {},
            "binding_constraints": [],
            "slacks": {},
            "shadow_prices": {},
        }

    tol = 1e-6
    slacks, binding, shadow_prices = {}, [], {}

    for name, con in zip(constraint_names, cvx_cons):
        try:
            lhs_val = float(con.args[0].value)
            rhs_val = float(con.args[1].value)
            gap = lhs_val - rhs_val
            slack = abs(gap) if con.__class__.__name__ == "Equality" else abs(gap)
            slacks[name] = round(slack, 6)
            if abs(gap) < tol:
                binding.append(name)
        except Exception:
            slacks[name] = None

        dual = con.dual_value
        shadow_prices[name] = round(float(dual), 6) if dual is not None else None

    return {
        "status": problem.status,
        "optimal_value": round(float(problem.value), 4),
        "variable_values": {
            name: round(float(cvx_vars[name].value), 6) for name in variable_names
        },
        "binding_constraints": binding,
        "slacks": slacks,
        "shadow_prices": shadow_prices,
    }

## 3. Two tools: expression solver + markdown writer

The first tool solves an LP from algebraic expressions.
The second tool writes the LP report markdown to disk.

In [4]:
@function_tool
def solve_lp_from_expressions(
    sense: Literal["maximize", "minimize"],
    variable_names: list[str],
    objective: str,
    constraints: list[str],
    constraint_names: list[str],
) -> dict:
    """Solve a continuous LP from algebraic-expression strings.

    Do not precompute coefficients. Use raw numbers and operators in expression form.
    Variables are non-negative by default.
    """
    return _solve_lp_from_exprs(
        sense=sense,
        variable_names=variable_names,
        objective=objective,
        constraints=constraints,
        constraint_names=constraint_names,
    )


@function_tool
def write_lp_report_markdown(
    output_filename: str,
    problem_summary: str,
    lp_type: str,
    decision_variables: list[str],
    objective_function: str,
    constraints: list[str],
    solver_status: str,
    optimal_objective_value: float,
    solution_variable_names: list[str],
    solution_variable_values: list[float],
    binding_constraints: list[str],
    business_recommendation: str,
) -> dict:
    """Write a markdown report for an LP model and its optimal solution.

    The report includes full formulation, final solution, and business recommendation.

    solution_variable_names and solution_variable_values must have the same length and
    be in the same order. Pass these as two parallel lists (no dict) so the tool's JSON
    schema stays strict-mode compatible.
    """
    if len(solution_variable_names) != len(solution_variable_values):
        raise ValueError("solution_variable_names and solution_variable_values must align in length.")

    if not output_filename.endswith(".md"):
        output_filename = f"{output_filename}.md"

    output_dir = Path.cwd() / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / output_filename

    variable_lines = "\n".join(
        f"- **{name}** = {value}"
        for name, value in zip(solution_variable_names, solution_variable_values)
    )
    constraint_lines = "\n".join([f"- {c}" for c in constraints])
    decision_var_lines = "\n".join([f"- {v}" for v in decision_variables])
    binding_text = ", ".join(binding_constraints) if binding_constraints else "None"

    md = f"""# LP Analysis Report

## Problem Summary
{problem_summary}

## LP Formulation
**Type:** {lp_type}

### Decision Variables
{decision_var_lines}

### Objective Function
- {objective_function}

### Constraints
{constraint_lines}

### Non-negativity
- All decision variables are constrained to be >= 0.

## Final Solution
- **Solver status:** {solver_status}
- **Optimal objective value:** {optimal_objective_value}

### Optimal Variable Values
{variable_lines}

### Binding Constraints
- {binding_text}

## Business Recommendation
{business_recommendation}
"""

    output_path.write_text(md, encoding="utf-8")

    return {
        "report_path": str(output_path),
        "message": "LP report markdown written successfully.",
    }


print("Tool 1:", solve_lp_from_expressions.name)
print("Tool 2:", write_lp_report_markdown.name)

Tool 1: solve_lp_from_expressions
Tool 2: write_lp_report_markdown


## 4. One agent, two tools, one workflow

The agent must do this in order:
1. formulate LP from text,
2. call `solve_lp_from_expressions`,
3. call `write_lp_report_markdown`.

In [5]:
WORKFLOW_INSTRUCTIONS = """
You are an operations-research assistant.

Follow this exact order for LP word problems:
1) Formulate a continuous LP (decision variables, objective, constraints, non-negativity).
2) Call solve_lp_from_expressions using algebraic expressions, not precomputed coefficients.
3) Read the tool result and then call write_lp_report_markdown.

For the solver tool:
- objective must be one expression string,
- constraints must be expression strings including <=, >=, or ==,
- keep units consistent throughout expressions.

For the writer tool:
- pass solution_variable_names and solution_variable_values as two parallel lists
  in the same order (do NOT pass a dict).

The markdown report must include:
- full LP formulation,
- final solution,
- business recommendation.

Never skip the writer tool. Never compare tools.
"""

lp_workflow_agent = Agent(
    name="LPWorkflowReporter",
    instructions=WORKFLOW_INSTRUCTIONS,
    model=MODEL,
    tools=[solve_lp_from_expressions, write_lp_report_markdown],
)

result = await Runner.run(lp_workflow_agent, PROBLEM_1)
print(result.final_output)

Optimal product mix:

- **A = 0**
- **B = 0**
- **C = 52.173913**
- **D = 78.26087**

### Maximum weekly profit
- **\$397.83**

### Interpretation
The best plan is to produce **only products C and D**.  
This mix uses up **Machine 1** and **Machine 3** completely, while **Machine 2** still has unused capacity.

### Recommendation
Focus production on **C and D** at the above levels, since they generate the highest weekly profit under the machine-time, material, and labor cost structure.


## 5. Inspect tool calls and validate the generated report

Always inspect `new_items` to confirm the workflow actually happened in the expected order.

In [6]:
import json

tool_sequence = []
report_path = None

for i, item in enumerate(result.new_items, 1):
    kind = type(item).__name__
    print(f"[{i}] {kind}")

    if isinstance(item, ToolCallItem):
        name = item.raw_item.name
        tool_sequence.append(name)
        print("   tool:", name)
        print("   args:", item.raw_item.arguments[:220], "...")

    if isinstance(item, ToolCallOutputItem):
        out = item.output if isinstance(item.output, str) else str(item.output)
        print("   output:", out[:220], "...")
        try:
            out_obj = item.output if isinstance(item.output, dict) else json.loads(item.output)
            if isinstance(out_obj, dict) and "report_path" in out_obj:
                report_path = out_obj["report_path"]
        except Exception:
            pass

    if isinstance(item, MessageOutputItem):
        text = "".join(c.text for c in item.raw_item.content if hasattr(c, "text"))
        print("   text:", text[:220], "...")

print("\nTool call order:", tool_sequence)

if report_path:
    p = Path(report_path)
    print("Report path:", p)
    if p.exists():
        report_text = p.read_text(encoding="utf-8")
        required_sections = [
            "## LP Formulation",
            "## Final Solution",
            "## Business Recommendation",
        ]
        checks = {sec: (sec in report_text) for sec in required_sections}
        print("Section checks:", checks)
        print("\nReport preview:\n")
        print(report_text[:1000])
    else:
        print("Report file was not found on disk.")
else:
    print("No report_path found in tool outputs.")

[1] MessageOutputItem
   text: Formulating the LP with weekly time capacities converted to minutes, then solving. ...
[2] ToolCallItem
   tool: solve_lp_from_expressions
   args: {"sense":"maximize","variable_names":["A","B","C","D"],"objective":"(5*A + 4*B + 5*C + 5*D) - (2*A + 2*B + 1*C + 1*D) - ((4/60)*(12*A + 7*B + 8*C + 10*D) + (4/60)*(8*A + 9*B + 4*C + 0*D) + (3/60)*(5*A + 10*B + 7*C + 3*D) ...
[3] ToolCallOutputItem
   output: {'status': 'optimal', 'optimal_value': 397.8261, 'variable_values': {'A': 0.0, 'B': 0.0, 'C': 52.173913, 'D': 78.26087}, 'binding_constraints': ['M1_time', 'M3_time'], 'slacks': {'M1_time': 0.0, 'M2_time': 2191.304348, ' ...
[4] ToolCallItem
   tool: write_lp_report_markdown
   args: {"output_filename":"lp_product_mix_report.md","problem_summary":"A plant produces four products (A, B, C, D) using three machines with weekly time limits. The goal is to choose production quantities to maximize weekly pr ...
[5] ToolCallOutputItem
   output: {'report_path': '/